# 职位信息爬取与分析系统
## 0. 环境依赖与初始化
导入所需模块，配置日志和绘图字体。

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import jieba
import jieba.analyse
import schedule
import time
import re
from datetime import datetime
import logging
from collections import Counter
import os
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

# 配置日志
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# 一、爬取网站 (Web Scraping)
包含 `JobSpider` 类，负责网页请求、反爬处理、页面解析、职位信息提取（薪资、学历、经验、地点等）。

In [ ]:
class JobSpider:
    def __init__(self):
        self.base_url = "http://www.job001.cn"
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        })
        self.job_categories = {
            '技术类': '/jobs/jishu/',
            '市场类': '/jobs/shichang/',
            '行政类': '/jobs/xingzheng/',
            '财务类': '/jobs/caiwu/',
            '设计类': '/jobs/sheji/'
        }
        # 初始化jieba分词，添加自定义词典
        self.init_jieba()

    def init_jieba(self):
        """初始化jieba分词器"""
        # 添加自定义词汇
        custom_words = [
            '前端开发', '后端开发', '数据分析', '市场营销',
            '视觉设计', '财务会计', 'UI设计', 'UX设计', '产品经理',
            '项目管理', 'Java开发', 'Python开发', 'Web前端', '移动端',
            '数据库', '服务器', '云计算', '人工智能', '机器学习'
        ]
        for word in custom_words:
            jieba.add_word(word)

    def parse_salary(self, salary_text):
        """解析薪资范围"""
        if not salary_text or '面议' in salary_text or '保密' in salary_text:
            return None, None

        try:
            # 清理文本
            salary_text = salary_text.replace('，', '').replace('k', '000').replace('K', '000')
            # 匹配数字模式
            numbers = re.findall(r'(\d+(?:\.\d+)?)', salary_text)
            numbers = [float(num) for num in numbers if float(num) > 0]

            if len(numbers) >= 2:
                min_salary = numbers[0] * 1000 if numbers[0] < 100 else numbers[0]
                max_salary = numbers[1] * 1000 if numbers[1] < 100 else numbers[1]
                return min_salary, max_salary
            elif len(numbers) == 1:
                salary = numbers[0] * 1000 if numbers[0] < 100 else numbers[0]
                return salary, salary
            else:
                return None, None
        except:
            return None, None

    def parse_education(self, text):
        """解析学历要求"""
        if not text:
            return '不限'
        edu_levels = ['博士', '硕士', '本科', '大专', '中专', '高中', '初中']
        for level in edu_levels:
            if level in text:
                return level
        return '不限'

    def parse_experience(self, text):
        """解析工作经验要求"""
        if not text:
            return '不限'
        if '不限' in text or '无经验' in text or '经验不限' in text:
            return '不限'
        elif '应届' in text or '毕业生' in text:
            return '应届生'
        elif '1年' in text or '1-3' in text or '1年以上' in text:
            return '1-3年'
        elif '3-5' in text or '3年以上' in text:
            return '3-5年'
        elif '5-10' in text or '5年以上' in text:
            return '5-10年'
        elif '10年' in text or '10年以上' in text:
            return '10年以上'
        else:
            return '不限'

    def parse_location(self, location_text):
        """解析工作地点"""
        if not location_text:
            return '未知', '未知'
        # 提取城市信息
        if '/' in location_text:
            parts = location_text.split('/')
            city = parts[0] if len(parts) > 0 else '未知'
            district = parts[1] if len(parts) > 1 else '未知'
        else:
            city = location_text
            district = '未知'
        return city, district

    def crawl_jobs_by_category(self, category, url_suffix, max_pages=2):
        """爬取指定类别的职位信息"""
        jobs = []
        for page in range(1, max_pages + 1):
            try:
                if page == 1:
                    url = f"{self.base_url}{url_suffix}"
                else:
                    url = f"{self.base_url}{url_suffix}index_{page}.html"

                logging.info(f"爬取 {category} 第{page}页: {url}")
                response = self.session.get(url, timeout=10)
                response.encoding = 'utf-8'

                if response.status_code != 200:
                    logging.warning(f"请求失败: {response.status_code}")
                    continue

                soup = BeautifulSoup(response.text, 'html.parser')
                job_list = soup.find_all('div', class_='jobs_Con')

                if not job_list:
                    logging.info(f"第{page}页没有找到职位信息")
                    break

                for job in job_list:
                    try:
                        job_data = self.parse_job_item(job, category)
                        if job_data:
                            jobs.append(job_data)
                    except Exception as e:
                        logging.error(f"解析职位信息失败: {e}")
                        continue

                time.sleep(0.5) # 防止请求过快
            except Exception as e:
                logging.error(f"爬取第{page}页失败: {e}")
                continue
        return jobs

    def parse_job_item(self, job_item, category):
        """解析单个职位信息"""
        try:
            # 职位名称
            job_name_elem = job_item.find('a', class_='jobNameCon')
            job_name = job_name_elem.text.strip() if job_name_elem else "未知职位"

            # 公司名称
            company_elem = job_item.find('span', class_='shortName')
            if not company_elem:
                company_elem = job_item.find(lambda tag: tag.name == 'span' and 'company' in tag.get('class', []))
            company = company_elem.text.strip() if company_elem else "未知公司"

            # 工作地点
            location_elem = job_item.find('span', class_='cityConJobswork')
            location = location_elem.text.strip() if location_elem else "未知地点"
            city, district = self.parse_location(location)

            # 薪资
            salary_elem = job_item.find('span', class_='salaryList')
            salary_text = salary_elem.text.strip() if salary_elem else ""
            min_salary, max_salary = self.parse_salary(salary_text)
            avg_salary = (min_salary + max_salary) / 2 if min_salary and max_salary else None

            # 解析工作经验和其他信息
            full_text = job_item.get_text()
            experience = self.parse_experience(full_text)
            education = self.parse_education(full_text)

            # 职位描述
            desc_elem = job_item.find('div', class_='jobsPopDes_text')
            if not desc_elem:
                desc_elem = job_item.find('div', class_='job_popBox')
            description = desc_elem.text.strip() if desc_elem else ""

            # 发布时间
            time_elem = job_item.find('span', class_='time')
            publish_time = time_elem.text.strip() if time_elem else datetime.now().strftime("%Y-%m-%d")

            job_data = {
                '职位名称': job_name, '公司名称': company, '工作地点': location,
                '城市': city, '区域': district, '薪资范围': salary_text,
                '最低薪资': min_salary, '最高薪资': max_salary, '平均薪资': avg_salary,
                '工作经验': experience, '学历要求': education, '职位描述': description,
                '发布时间': publish_time, '职位类别': category,
                '爬取时间': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            }
            return job_data
        except Exception as e:
            logging.error(f"解析职位项失败: {e}")
            return None

    def crawl_all_jobs(self):
        """爬取所有类别的职位信息"""
        all_jobs = []
        for category, url_suffix in self.job_categories.items():
            logging.info(f"开始爬取 {category} 职位...")
            jobs = self.crawl_jobs_by_category(category, url_suffix, max_pages=2)
            all_jobs.extend(jobs)
            logging.info(f"{category} 爬取完成，共获取 {len(jobs)} 个职位")
            time.sleep(1)
        return all_jobs

# 二、数据预处理 (Data Preprocessing)
定义 `JobAnalyzer` 类，并进行数据去重、空值填充、异常薪资过滤等操作。

In [ ]:
class JobAnalyzer:
    def __init__(self, jobs_df):
        self.df = jobs_df
        self.clean_data()

    # ================= 数据预处理 =================
    def clean_data(self):
        self.df = self.df.drop_duplicates(subset=['职位名称', '公司名称', '工作地点'], keep='first')
        if '平均薪资' in self.df.columns:
            self.df = self.df[self.df['平均薪资'].notna()]
            self.df = self.df[(self.df['平均薪资'] > 1000) & (self.df['平均薪资'] < 100000)]

        self.df['工作经验'] = self.df['工作经验'].fillna('不限')
        self.df['学历要求'] = self.df['学历要求'].fillna('不限')
        self.df['城市'] = self.df['城市'].fillna('未知')

    # ================= 数据可视化分析 =================
    def create_visualizations(self):
        """创建多种可视化图表"""
        if self.df.empty:
            print("没有数据可分析")
            return

        plt.figure(figsize=(20, 15))

        # 1. 柱状图：各职位类别平均薪资对比
        plt.subplot(3, 3, 1)
        if '平均薪资' in self.df.columns and not self.df['平均薪资'].isna().all():
            salary_by_category = self.df.groupby('职位类别')['平均薪资'].mean().sort_values(ascending=False)
            salary_by_category.plot(kind='bar', color='skyblue')
            plt.title('各职位类别平均薪资对比', fontsize=14, fontweight='bold')
            plt.ylabel('平均薪资(元)')
            plt.xticks(rotation=45)
        else:
            plt.text(0.5, 0.5, '无薪资数据', ha='center', va='center', transform=plt.gca().transAxes)
            plt.title('各职位类别平均薪资对比', fontsize=14, fontweight='bold')

        # 2. 饼图：学历要求分布
        plt.subplot(3, 3, 2)
        edu_dist = self.df['学历要求'].value_counts()
        if len(edu_dist) > 0:
            plt.pie(edu_dist.values, labels=edu_dist.index, autopct='%1.1f%%', startangle=90)
            plt.title('学历要求分布', fontsize=14, fontweight='bold')
        else:
            plt.text(0.5, 0.5, '无学历数据', ha='center', va='center', transform=plt.gca().transAxes)
            plt.title('学历要求分布', fontsize=14, fontweight='bold')

        # 3. 折线图：各城市岗位数量趋势
        plt.subplot(3, 3, 3)
        city_jobs = self.df['城市'].value_counts().head(10)
        if len(city_jobs) > 0:
            plt.plot(city_jobs.index, city_jobs.values, marker='o', linewidth=2, markersize=8)
            plt.title('各城市岗位数量TOP10', fontsize=14, fontweight='bold')
            plt.xticks(rotation=45)
            plt.ylabel('岗位数量')
        else:
            plt.text(0.5, 0.5, '无城市数据', ha='center', va='center', transform=plt.gca().transAxes)
            plt.title('各城市岗位数量TOP10', fontsize=14, fontweight='bold')

        # 4. 散点图：工作经验与薪资关系
        plt.subplot(3, 3, 4)
        if '平均薪资' in self.df.columns and not self.df['平均薪资'].isna().all():
            experience_order = ['不限', '应届生', '1-3年', '3-5年', '5-10年', '10年以上']
            available_experience = [exp for exp in experience_order if exp in self.df['工作经验'].values]
            if available_experience:
                exp_salary = self.df.groupby('工作经验')['平均薪资'].mean().reindex(available_experience)
                plt.scatter(range(len(exp_salary)), exp_salary.values, s=100, alpha=0.6, c='red')
                plt.plot(range(len(exp_salary)), exp_salary.values, linestyle='--', alpha=0.5)
                plt.title('工作经验与薪资关系', fontsize=14, fontweight='bold')
                plt.xticks(range(len(exp_salary)), exp_salary.index, rotation=45)
                plt.ylabel('平均薪资(元)')
            else:
                plt.text(0.5, 0.5, '无经验薪资数据', ha='center', va='center', transform=plt.gca().transAxes)
                plt.title('工作经验与薪资关系', fontsize=14, fontweight='bold')
        else:
            plt.text(0.5, 0.5, '无薪资数据', ha='center', va='center', transform=plt.gca().transAxes)
            plt.title('工作经验与薪资关系', fontsize=14, fontweight='bold')

        # 5. 热力图：职位类别与学历要求的交叉分析
        plt.subplot(3, 3, 5)
        cross_tab = pd.crosstab(self.df['职位类别'], self.df['学历要求'])
        if not cross_tab.empty:
            sns.heatmap(cross_tab, annot=True, fmt='d', cmap='YlOrRd')
            plt.title('职位类别-学历要求热力图', fontsize=14, fontweight='bold')
            plt.xticks(rotation=45)
            plt.yticks(rotation=0)
        else:
            plt.text(0.5, 0.5, '无交叉分析数据', ha='center', va='center', transform=plt.gca().transAxes)
            plt.title('职位类别-学历要求热力图', fontsize=14, fontweight='bold')

        # 6. 箱线图：各职位类别薪资分布
        plt.subplot(3, 3, 6)
        if '平均薪资' in self.df.columns and not self.df['平均薪资'].isna().all():
            categories = self.df['职位类别'].unique()
            category_data = [self.df[self.df['职位类别'] == cat]['平均薪资'].dropna() for cat in categories]
            if any(len(data) > 0 for data in category_data):
                plt.boxplot([data for data in category_data if len(data) > 0],
                            labels=[cat for i, cat in enumerate(categories) if len(category_data[i]) > 0])
                plt.title('各职位类别薪资分布箱线图', fontsize=14, fontweight='bold')
                plt.xticks(rotation=45)
                plt.ylabel('薪资(元)')
            else:
                plt.text(0.5, 0.5, '无薪资分布数据', ha='center', va='center', transform=plt.gca().transAxes)
                plt.title('各职位类别薪资分布箱线图', fontsize=14, fontweight='bold')
        else:
            plt.text(0.5, 0.5, '无薪资数据', ha='center', va='center', transform=plt.gca().transAxes)
            plt.title('各职位类别薪资分布箱线图', fontsize=14, fontweight='bold')

        # 7. 堆叠柱状图：各城市不同职位类别分布
        plt.subplot(3, 3, 7)
        top_cities = self.df['城市'].value_counts().head(5).index
        if len(top_cities) > 0:
            city_category = self.df[self.df['城市'].isin(top_cities)].groupby(['城市', '职位类别']).size().unstack(fill_value=0)
            if not city_category.empty:
                city_category.plot(kind='bar', stacked=True, ax=plt.gca())
                plt.title('TOP5城市职位类别分布', fontsize=14, fontweight='bold')
                plt.xticks(rotation=45)
                plt.ylabel('岗位数量')
                plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
            else:
                plt.text(0.5, 0.5, '无城市类别数据', ha='center', va='center', transform=plt.gca().transAxes)
                plt.title('TOP5城市职位类别分布', fontsize=14, fontweight='bold')
        else:
            plt.text(0.5, 0.5, '无城市数据', ha='center', va='center', transform=plt.gca().transAxes)
            plt.title('TOP5城市职位类别分布', fontsize=14, fontweight='bold')

        # 8. 水平柱状图：高频技能关键词
        plt.subplot(3, 3, 8)
        keywords = self.extract_keywords()[:10]
        if keywords:
            keywords_df = pd.DataFrame(keywords, columns=['关键词', '频次'])
            plt.barh(range(len(keywords_df)), keywords_df['频次'])
            plt.yticks(range(len(keywords_df)), keywords_df['关键词'])
            plt.title('职位关键词TOP10', fontsize=14, fontweight='bold')
            plt.xlabel('出现频次')
        else:
            plt.text(0.5, 0.5, '无关键词数据', ha='center', va='center', transform=plt.gca().transAxes)
            plt.title('职位关键词TOP10', fontsize=14, fontweight='bold')

        # 9. 职位类别分布饼图
        plt.subplot(3, 3, 9)
        category_dist = self.df['职位类别'].value_counts()
        if len(category_dist) > 0:
            plt.pie(category_dist.values, labels=category_dist.index, autopct='%1.1f%%', startangle=90)
            plt.title('职位类别分布', fontsize=14, fontweight='bold')
        else:
            plt.text(0.5, 0.5, '无类别数据', ha='center', va='center', transform=plt.gca().transAxes)
            plt.title('职位类别分布', fontsize=14, fontweight='bold')

        plt.tight_layout()
        plt.savefig('job_analysis.png', dpi=300, bbox_inches='tight')
        plt.show()

    def create_wordcloud(self):
        """生成词云图"""
        try:
            text = ' '.join(self.df['职位名称'].fillna('').astype(str) + ' ' +
                            self.df['职位描述'].fillna('').astype(str))
            if not text.strip():
                print("没有文本数据生成词云")
                return
            wordcloud = WordCloud(
                font_path='simhei.ttf', width=800, height=400,
                background_color='white', max_words=100, collocations=False
            ).generate(text)
            plt.figure(figsize=(12, 6))
            plt.imshow(wordcloud, interpolation='bilinear')
            plt.axis('off')
            plt.title('职位信息词云图', fontsize=16, fontweight='bold')
            plt.tight_layout()
            plt.savefig('job_wordcloud.png', dpi=300, bbox_inches='tight')
            plt.show()
        except Exception as e:
            logging.error(f"生成词云失败: {e}")

    # ================= 关键词提取 =================
    def extract_keywords(self, top_k=50):
        """提取职位关键词"""
        try:
            # 合并职位名称和描述
            text = ' '.join(self.df['职位名称'].fillna('').astype(str) + ' ' +
                            self.df['职位描述'].fillna('').astype(str))
            if not text.strip():
                return []
            # 使用jieba提取关键词
            keywords = jieba.analyse.extract_tags(text, topK=top_k, withWeight=True)
            # 过滤掉一些无意义的词
            stop_words = {'公司', '工作', '职位', '要求', '经验', '能力', '负责', '相关', '以上', '具备', '岗位', '描述'}
            filtered_keywords = [(word, freq) for word, freq in keywords
                                 if word not in stop_words and len(word) > 1]
            return filtered_keywords
        except Exception as e:
            logging.error(f"关键词提取失败: {e}")
            return []

    def generate_report(self):
        """生成分析报告"""
        try:
            report = {
                '总职位数': len(self.df),
                '涉及城市数': self.df['城市'].nunique(),
                '职位类别数': self.df['职位类别'].nunique(),
                '最热门职位类别': self.df['职位类别'].mode().iloc[0] if not self.df['职位类别'].mode().empty else '无数据',
                '最常见学历要求': self.df['学历要求'].mode().iloc[0] if not self.df['学历要求'].mode().empty else '无数据',
                '最常见工作经验': self.df['工作经验'].mode().iloc[0] if not self.df['工作经验'].mode().empty else '无数据'
            }
            if '平均薪资' in self.df.columns and not self.df['平均薪资'].isna().all():
                report['平均薪资'] = f"{self.df['平均薪资'].mean():.2f}元"
                max_salary_idx = self.df['平均薪资'].idxmax()
                report['最高薪资职位'] = f"{self.df.loc[max_salary_idx]['职位名称']} ({self.df.loc[max_salary_idx]['平均薪资']:.2f}元)"
            else:
                report['平均薪资'] = '无数据'
                report['最高薪资职位'] = '无数据'

            print("\n=== 职位数据分析报告 ===")
            for key, value in report.items():
                print(f"{key}: {value}")

            keywords = self.extract_keywords(10)
            if keywords:
                print("\n=== 热门技能关键词TOP10 ===")
                for i, (word, freq) in enumerate(keywords[:10], 1):
                    print(f"{i}. {word}: {freq:.4f}")
            return report
        except Exception as e:
            logging.error(f"生成报告失败: {e}")
            return {}

# 五、数据更新机制 (Data Update Mechanism)
包含 `JobMonitor` 类，负责调用爬虫、合并新旧数据、去重以及定时任务调度。

In [ ]:
class JobMonitor:
    def __init__(self):
        self.spider = JobSpider()
        self.data_file = 'job_data.csv'

    def update_data(self):
        """更新数据"""
        logging.info("开始更新职位数据...")
        try:
            new_jobs = self.spider.crawl_all_jobs()
            new_df = pd.DataFrame(new_jobs)
            if not new_df.empty:
                # 保存新数据
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                new_df.to_csv(f'job_data_{timestamp}.csv', index=False, encoding='utf-8-sig')
                # 更新主数据文件
                try:
                    existing_df = pd.read_csv(self.data_file)
                    combined_df = pd.concat([existing_df, new_df], ignore_index=True)
                    combined_df = combined_df.drop_duplicates(
                        subset=['职位名称', '公司名称', '工作地点', '发布时间'], keep='last'
                    )
                except FileNotFoundError:
                    combined_df = new_df

                combined_df.to_csv(self.data_file, index=False, encoding='utf-8-sig')
                logging.info(f"数据更新完成。新增 {len(new_df)} 条记录，总记录数: {len(combined_df)}")
                return self.analyze_data(combined_df)
            else:
                logging.warning("未获取到新数据")
                return None
        except Exception as e:
            logging.error(f"数据更新失败: {e}")
            return None

    def analyze_data(self, df):
        """分析数据"""
        try:
            analyzer = JobAnalyzer(df)
            analyzer.create_visualizations()
            analyzer.create_wordcloud()
            report = analyzer.generate_report()

            keywords = analyzer.extract_keywords()
            if keywords:
                keywords_df = pd.DataFrame(keywords, columns=['关键词', '权重'])
                keywords_df.to_csv('job_keywords.csv', index=False, encoding='utf-8-sig')
            return analyzer
        except Exception as e:
            logging.error(f"数据分析失败: {e}")
            return None

    def schedule_updates(self, interval_hours=1):
        """设置定时任务"""
        schedule.every(interval_hours).hours.do(self.update_data)
        logging.info(f"定时任务已启动，每{interval_hours}小时更新一次数据...")
        while True:
            try:
                schedule.run_pending()
                time.sleep(60)
            except KeyboardInterrupt:
                logging.info("定时任务已停止")
                break
            except Exception as e:
                logging.error(f"定时任务执行失败: {e}")
                time.sleep(60)

# 六、主程序 (Main Program)
提供命令行交互菜单，控制数据更新、定时任务和数据分析。

In [ ]:
def main():
    """主函数"""
    monitor = JobMonitor()
    print("=== 职位信息爬取与分析系统 ===")
    print("1. 立即更新数据并分析")
    print("2. 启动定时监控")
    print("3. 分析现有数据")

    choice = input("请选择操作 (1/2/3): ").strip()

    if choice == '1':
        analyzer = monitor.update_data()
        if analyzer is None:
            print("数据更新失败，尝试分析现有数据...")
            try:
                df = pd.read_csv(monitor.data_file)
                analyzer = monitor.analyze_data(df)
            except FileNotFoundError:
                print("没有找到现有数据文件")
    elif choice == '2':
        interval = input("请输入更新间隔(小时，默认1小时): ").strip()
        try:
            interval = float(interval) if interval else 1
            monitor.schedule_updates(interval)
        except ValueError:
            print("输入无效，使用默认间隔1小时")
            monitor.schedule_updates(1)
    elif choice == '3':
        try:
            df = pd.read_csv(monitor.data_file)
            analyzer = monitor.analyze_data(df)
        except FileNotFoundError:
            print("没有找到现有数据文件，请先运行选项1获取数据")
    else:
        print("无效选择，退出程序")

if __name__ == "__main__":
    main()